# LearnOS quickstart

Run the cells top to bottom. The desktop appears under cell 1 and stays live: when later cells act,
watch it update. You can click around it yourself too (dock at the bottom, drag windows by their title bars).

In [ ]:
# 1. Start LearnOS inside this notebook and show its desktop
from learnos_client import start, show, show_in_tab, make_tools
env = start()
show(height=680)

## 2. Start an episode
An *instance* is a task setup: the student's laptop contents, a simulated student, and a budget.
The menu bar on the desktop should now show `friday-build-01`.

In [ ]:
import json
from pathlib import Path
instance = json.loads(Path("../instances/friday-build-01.json").read_text())
obs = env.reset(instance)
print(instance["instruction"])

## 3. Act like an agent, by hand
Each step is one tool call, the same calls an AI agent makes. Scroll up to the desktop while this runs:
with *Follow agent* on, it opens the window for each action.

In [ ]:
import time
R = "/course/readings/week3/"
steps = [
    ("messages_read_channel", {"channel": "#course"}),
    ("feed_scroll",           {"n": 6}),
    ("reader_open_section",   {"path": R + "environments.md", "section": "Partial observability (POMDP)", "concept": "pomdp"}),
    ("quiz_run",              {"concept": "pomdp", "n_items": 3, "difficulty": 0.5}),
    ("session_wait",          {"minutes": 20}),
    ("calendar_add_block",    {"title": "Review pass^k", "start": 200, "duration": 45, "concept": "pass_k"}),
]
for action, args in steps:
    out = env.step(action, args)
    print(f"#{out['step']} {action}\n   {out['output'][:160]}\n")
    time.sleep(1.5)

## 4. Look behind the curtain (grader)
The agent never sees this. You do, through the grader token `start()` gave this client.

In [ ]:
env.step("session_end", {"summary": "quickstart done"})
print("proxies:", env.grader("proxies"))
print("hidden student state:", {k: v for k, v in env.grader("true_state").items() if k in ("p_know", "attention", "motivation", "feed_minutes", "phone_minutes")})

## 5. Optional: a real AI agent
Needs an API key (`OPENAI_API_KEY`). Reset, then let the agent drive while you watch the desktop.

In [ ]:
import os
if os.getenv("OPENAI_API_KEY"):
    from smolagents import OpenAIServerModel, ToolCallingAgent
    agent = ToolCallingAgent(tools=make_tools(env), model=OpenAIServerModel(model_id="gpt-5.4-mini"), max_steps=15)
    env.reset(instance)
    agent.run(instance["instruction"])
else:
    print("Set OPENAI_API_KEY to run a real agent.")